In [11]:
import numpy as np
import pandas as pd

In [12]:
df = pd.read_csv("../data/restaurants_data.csv")

In [13]:
df.head()

,Name,Location,Cuisine,Rating,Seating Capacity,Average Meal Price,Marketing Budget,Social Media Followers,Chef Experience Years,Number of Reviews,Avg Review Length,Ambience Score,Service Quality Score,Parking Availability,Weekend Reservations,Weekday Reservations,Revenue
0,Restaurant 0,Rural,Japanese,4.0,38,73.98,2224,23406,13,185,161.924906,1.3,7.0,Yes,13,4,638945.52
1,Restaurant 1,Downtown,Mexican,3.2,76,28.11,4416,42741,8,533,148.759717,2.6,3.4,Yes,48,6,490207.83
2,Restaurant 2,Rural,Italian,4.7,48,48.29,2796,37285,18,853,56.849189,5.3,6.7,No,27,14,541368.62
3,Restaurant 3,Rural,Italian,4.4,34,51.55,1167,15214,13,82,205.433265,4.6,2.8,Yes,9,17,404556.80
4,Restaurant 4,Downtown,Japanese,4.9,88,75.98,3639,40171,9,78,241.681584,8.6,2.1,No,37,26,1491046.35


In [14]:
df.isnull().sum()

Name                      0
Location                  0
Cuisine                   0
Rating                    0
Seating Capacity          0
Average Meal Price        0
Marketing Budget          0
Social Media Followers    0
Chef Experience Years     0
Number of Reviews         0
Avg Review Length         0
Ambience Score            0
Service Quality Score     0
Parking Availability      0
Weekend Reservations      0
Weekday Reservations      0
Revenue                   0
dtype: int64

In [15]:
print(df.duplicated().sum())

0


In [16]:
df["Parking Availability"].unique()

array(['Yes', 'No'], dtype=object)

In [17]:
df["Cuisine"].unique()

array(['Japanese', 'Mexican', 'Italian', 'Indian', 'French', 'American'],
      dtype=object)

In [18]:
df["Location"].unique()

array(['Rural', 'Downtown', 'Suburban'], dtype=object)

In [19]:
df = df.drop(columns="Name")

In [20]:
df.head()

,Location,Cuisine,Rating,Seating Capacity,Average Meal Price,Marketing Budget,Social Media Followers,Chef Experience Years,Number of Reviews,Avg Review Length,Ambience Score,Service Quality Score,Parking Availability,Weekend Reservations,Weekday Reservations,Revenue
0,Rural,Japanese,4.0,38,73.98,2224,23406,13,185,161.924906,1.3,7.0,Yes,13,4,638945.52
1,Downtown,Mexican,3.2,76,28.11,4416,42741,8,533,148.759717,2.6,3.4,Yes,48,6,490207.83
2,Rural,Italian,4.7,48,48.29,2796,37285,18,853,56.849189,5.3,6.7,No,27,14,541368.62
3,Rural,Italian,4.4,34,51.55,1167,15214,13,82,205.433265,4.6,2.8,Yes,9,17,404556.80
4,Downtown,Japanese,4.9,88,75.98,3639,40171,9,78,241.681584,8.6,2.1,No,37,26,1491046.35


In [21]:
X = df.drop("Revenue",axis=1)
y = df["Revenue"]

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [23]:
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

In [24]:
num_cols

['Rating',
 'Seating Capacity',
 'Average Meal Price',
 'Marketing Budget',
 'Social Media Followers',
 'Chef Experience Years',
 'Number of Reviews',
 'Avg Review Length',
 'Ambience Score',
 'Service Quality Score',
 'Weekend Reservations',
 'Weekday Reservations']

In [25]:
cat_cols

['Location', 'Cuisine', 'Parking Availability']

In [26]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder",OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [27]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline,num_cols),
        ("cat", cat_pipeline,cat_cols)
    ],
    remainder="passthrough"
)

In [28]:
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [29]:
models = {
    "LinearRegression":LinearRegression(),
    "DecisionTreeRegressor":DecisionTreeRegressor(),
    "RandomForestRegressor":RandomForestRegressor(),
    "GradientBoostingRegressor":GradientBoostingRegressor(),
    "KNeighborsRegressor":KNeighborsRegressor(n_neighbors=5),
    "Ridge":Ridge()
}

In [30]:
def evaluate_train(model, X_train, y_train):

    y_train_pred = model.predict(X_train)

    mae = mean_absolute_error(y_train, y_train_pred)
    rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    r2 = r2_score(y_train, y_train_pred)

    print("Training Performance")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

    return r2, mae, rmse

In [31]:
def evaluate_test(model, X_test, y_test):

    y_test_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)

    print("Testing Performance")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

    return r2, mae, rmse

In [32]:
best_model = None
best_score = float("-inf")

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train model
    pipeline.fit(X_train, y_train)

    print("=" * 50)
    print(f"Model: {name}\n")

    # Evaluate training data
    train_r2, train_mae, train_rmse = evaluate_train(
        pipeline, X_train, y_train
    )

    print()

    # Evaluate testing data
    test_r2, test_mae, test_rmse = evaluate_test(
        pipeline, X_test, y_test
    )

    # Select best model based on test R² score
    if test_r2 > best_score:
        best_model = name
        best_score = test_r2
        best_pipeline = pipeline


print("\n" + "=" * 50)
print("Best Model:", best_model)
print("Best Test R² Score:", best_score)

Model: LinearRegression

Training Performance
R² Score: 0.9587
MAE: 41187.7704
RMSE: 54350.4765

Testing Performance
R² Score: 0.9554
MAE: 43217.2498
RMSE: 56511.9353


Model: DecisionTreeRegressor

Training Performance
R² Score: 1.0000
MAE: 0.0000
RMSE: 0.0000

Testing Performance
R² Score: 0.9968
MAE: 11646.7832
RMSE: 15256.3455
Model: RandomForestRegressor

Training Performance
R² Score: 0.9999
MAE: 2333.8877
RMSE: 3040.1802

Testing Performance
R² Score: 0.9992
MAE: 5886.4211
RMSE: 7541.9553
Model: GradientBoostingRegressor

Training Performance
R² Score: 0.9988
MAE: 6968.8102
RMSE: 9100.3306

Testing Performance
R² Score: 0.9986
MAE: 7710.8964
RMSE: 9946.1588
Model: KNeighborsRegressor

Training Performance
R² Score: 0.9442
MAE: 49466.1173
RMSE: 63140.6730

Testing Performance
R² Score: 0.9249
MAE: 57831.1718
RMSE: 73372.2331
Model: Ridge

Training Performance
R² Score: 0.9587
MAE: 41183.2519
RMSE: 54350.7444

Testing Performance
R² Score: 0.9555
MAE: 43201.6063
RMSE: 56502.0857

Best Model: RandomForestRegressor
Best Test R² Score: 0.9992063674269709


In [33]:
model = RandomForestRegressor()

best_model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

best_model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Rating', 'Seating Capacity',
                                                   'Average Meal Price',
                                                   'Marketing Budget',
                                                   'Social Media Followers',
                                                   'Chef Experience Years',
                                                   'Number of Reviews',
                                                   'Avg Review Length',
                                                   'Ambience Score',
                                                   'Service Quality Score',
                                                   'Weekend Reservations',
                                                   'Weekday Reservations']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Location', 'Cuisine',
                                                   'Parking Availability'])])),
                ('model', RandomForestRegressor())])

In [34]:
prediction = best_model_pipeline.predict(X_test)
best_model_score = r2_score(y_test,prediction)
print("Best Model Score :",best_model_score)

Best Model Score : 0.9992056961790787


In [36]:
import pickle

with open("../model/pipeline.pkl", "wb") as file:
    pickle.dump(pipeline, file)
    print("File saved")

File saved
